# TP: Agentes Reactivos para Refuerzo de Taxis

**Alumno:** Agustín Beade  

Este notebook implementa y compara dos agentes reactivos que recomiendan si se deben reforzar taxis en una zona de Nueva York:

- **Parte 1** – Agente reactivo simple (sin memoria)
- **Parte 2** – Agente reactivo basado en modelo (con estado interno)
- **Parte 3** – Bitácora comparativa sobre un escenario real
- **Parte 4** – Tests obligatorios
- **Parte 5** – Informe: preguntas, PEAS y limitaciones

## 🔧 Setup: Instalación de dependencias y clonado del repo

In [ ]:
# Instalar dependencias necesarias para el simulador
!pip install pyarrow geopandas -q

# Clonar el repositorio para tener acceso al simulador
import os
REPO = "https://github.com/Agustin2102/clases_ds_ia.git"
BRANCH = "AgustinBeade"

if not os.path.exists("clases_ds_ia"):
    !git clone --branch {BRANCH} {REPO} clases_ds_ia
else:
    print("El repo ya está clonado.")

# Directorio de trabajo
LAB_DIR = "clases_ds_ia/LABORATORIOS/MOVILIDAD"
print(f"Directorio del laboratorio: {LAB_DIR}")

## Parte 1 y 2: Implementación de los Agentes

In [ ]:
from __future__ import annotations
import math
from typing import Any
import pandas as pd

ACCIONES = {"NO_REFORZAR", "RECOMENDAR_REFUERZO", "ABSTENERSE"}
UMBRAL_PRESION = 0.85

# ── VALIDACIÓN ──────────────────────────────────────────────────────────────
def _validar_percepcion(percepcion: dict[str, Any]) -> bool:
    """Verifica si la percepción contiene datos válidos para decidir."""
    if not isinstance(percepcion, dict):
        return False
    if "presion" not in percepcion or "capacidad_x" not in percepcion:
        return False
    capacidad = percepcion["capacidad_x"]
    presion   = percepcion["presion"]
    if isinstance(capacidad, bool) or isinstance(presion, bool):
        return False
    if not isinstance(capacidad, (int, float)) or not isinstance(presion, (int, float)):
        return False
    if math.isnan(presion) or math.isinf(presion) or presion < 0:
        return False
    if math.isnan(capacidad) or math.isinf(capacidad) or capacidad <= 0:
        return False
    return True

# ── PARTE 1: AGENTE REACTIVO SIMPLE ─────────────────────────────────────────
# Solo mira la percepción ACTUAL. No tiene memoria de horas anteriores.
# Regla: presion >= 0.85 → RECOMENDAR_REFUERZO, sino → NO_REFORZAR

def decidir_reactivo_simple(percepcion: dict[str, Any]) -> tuple[str, str]:
    """Devuelve (accion, motivo) usando solo la percepción actual."""
    if not _validar_percepcion(percepcion):
        return ("ABSTENERSE", "Percepción inválida o datos faltantes")
    presion = percepcion["presion"]
    if presion >= UMBRAL_PRESION:
        return ("RECOMENDAR_REFUERZO", f"Presion alta observada ({presion:.2f} >= {UMBRAL_PRESION})")
    return ("NO_REFORZAR", f"Presión dentro de limites normales ({presion:.2f} < {UMBRAL_PRESION})")

# ── PARTE 2: AGENTE REACTIVO BASADO EN MODELO ───────────────────────────────
# Mantiene un estado interno (memoria). Solo recomienda si la presión lleva
# 2 o más horas CONSECUTIVAS siendo alta.

def crear_estado_inicial() -> dict[str, Any]:
    """Crea el estado persistente inicial del agente basado en modelo."""
    return {
        "percepcion_valida": False,  # ¿El último dato era válido?
        "racha_presion_alta": 0,     # ¿Cuántas horas SEGUIDAS con presión alta?
        "presion_anterior": None,    # ¿Cuál fue la presión de la hora anterior?
        "ultima_accion": None,       # ¿Qué acción decidió la última vez?
    }

def actualizar_estado(estado_anterior: dict[str, Any], percepcion: dict[str, Any]) -> dict[str, Any]:
    """Actualiza el estado interno con la nueva percepción."""
    nuevo_estado = estado_anterior.copy()
    if not _validar_percepcion(percepcion):
        nuevo_estado["percepcion_valida"] = False
        nuevo_estado["racha_presion_alta"] = 0
        nuevo_estado["presion_anterior"] = None
        return nuevo_estado
    presion_actual = percepcion["presion"]
    nuevo_estado["percepcion_valida"] = True
    if presion_actual >= UMBRAL_PRESION:
        nuevo_estado["racha_presion_alta"] = estado_anterior.get("racha_presion_alta", 0) + 1
    else:
        nuevo_estado["racha_presion_alta"] = 0
    nuevo_estado["presion_anterior"] = presion_actual
    return nuevo_estado

def decidir_reactivo_modelo(estado_actual: dict[str, Any]) -> tuple[str, str]:
    """Devuelve (accion, motivo) a partir del estado interno actualizado."""
    if not estado_actual.get("percepcion_valida", False):
        return ("ABSTENERSE", "Percepción invalida o datos faltantes en el estado.")
    racha = estado_actual.get("racha_presion_alta", 0)
    if racha >= 2:
        return ("RECOMENDAR_REFUERZO", f"Presión alta persistente durante {racha} horas consecutivas.")
    return ("NO_REFORZAR", f"Presión alta durante {racha} hora(s); se requieren al menos 2 horas consecutivas.")

# ── PARTE 3: BITÁCORA ────────────────────────────────────────────────────────
def procesar_secuencia(percepciones: pd.DataFrame) -> pd.DataFrame:
    """Ejecuta ambos agentes en orden temporal y construye la bitácora comparativa."""
    estado_modelo = crear_estado_inicial()
    registros = []
    for _, fila in percepciones.iterrows():
        percepcion_dict = fila.to_dict()
        accion_simple, motivo_simple = decidir_reactivo_simple(percepcion_dict)
        estado_modelo = actualizar_estado(estado_modelo, percepcion_dict)
        accion_modelo, motivo_modelo = decidir_reactivo_modelo(estado_modelo)
        estado_modelo["ultima_accion"] = accion_modelo
        registros.append({
            "hora": percepcion_dict.get("hora"),
            "presion": percepcion_dict.get("presion"),
            "racha_presion_alta": estado_modelo.get("racha_presion_alta"),
            "accion_simple": accion_simple,
            "motivo_simple": motivo_simple,
            "accion_modelo": accion_modelo,
            "motivo_modelo": motivo_modelo,
        })
    return pd.DataFrame(registros)

print("✅ Agentes definidos correctamente.")

## Parte 4: Tests obligatorios

In [ ]:
def test_presion_baja():
    """Caso 1: Presión baja → ambos agentes devuelven NO_REFORZAR."""
    p = {"capacidad_x": 20, "presion": 0.50}
    accion_s, _ = decidir_reactivo_simple(p)
    assert accion_s == "NO_REFORZAR", f"Simple devolvió {accion_s}"
    estado = crear_estado_inicial()
    estado = actualizar_estado(estado, p)
    accion_m, _ = decidir_reactivo_modelo(estado)
    assert accion_m == "NO_REFORZAR", f"Modelo devolvió {accion_m}"
    print("✅ test_presion_baja OK")

def test_primera_hora_presion_alta():
    """Caso 2: 1ra hora alta → Simple recomienda, Modelo NO (espera confirmación)."""
    p = {"capacidad_x": 20, "presion": 0.90}
    accion_s, _ = decidir_reactivo_simple(p)
    assert accion_s == "RECOMENDAR_REFUERZO"
    estado = crear_estado_inicial()
    estado = actualizar_estado(estado, p)
    accion_m, _ = decidir_reactivo_modelo(estado)
    assert accion_m == "NO_REFORZAR"
    print("✅ test_primera_hora_presion_alta OK")

def test_segunda_hora_consecutiva_presion_alta():
    """Caso 3: 2da hora seguida alta → ambos recomiendan refuerzo."""
    p1 = {"capacidad_x": 20, "presion": 0.90}
    p2 = {"capacidad_x": 20, "presion": 0.88}
    estado = crear_estado_inicial()
    estado = actualizar_estado(estado, p1)
    estado = actualizar_estado(estado, p2)
    assert estado["racha_presion_alta"] == 2
    accion_m, _ = decidir_reactivo_modelo(estado)
    assert accion_m == "RECOMENDAR_REFUERZO"
    print("✅ test_segunda_hora_consecutiva_presion_alta OK")

def test_demostracion_dependencia_historia():
    """Prueba decisiva: misma percepción final, distintas historias → el modelo difiere."""
    p_baja = {"capacidad_x": 20, "presion": 0.50}
    p_alta = {"capacidad_x": 20, "presion": 0.90}
    # Historia A: [Baja, Alta]
    estado_a = crear_estado_inicial()
    estado_a = actualizar_estado(estado_a, p_baja)
    estado_a = actualizar_estado(estado_a, p_alta)
    accion_simple_a, _ = decidir_reactivo_simple(p_alta)
    accion_modelo_a, _ = decidir_reactivo_modelo(estado_a)
    # Historia B: [Alta, Alta]
    estado_b = crear_estado_inicial()
    estado_b = actualizar_estado(estado_b, p_alta)
    estado_b = actualizar_estado(estado_b, p_alta)
    accion_simple_b, _ = decidir_reactivo_simple(p_alta)
    accion_modelo_b, _ = decidir_reactivo_modelo(estado_b)
    # El simple reacciona igual (sin memoria)
    assert accion_simple_a == accion_simple_b == "RECOMENDAR_REFUERZO"
    # El modelo reacciona diferente (con memoria)
    assert accion_modelo_a == "NO_REFORZAR"
    assert accion_modelo_b == "RECOMENDAR_REFUERZO"
    print("✅ test_demostracion_dependencia_historia OK")

def test_percepcion_invalida():
    """Ambos responden ABSTENERSE ante datos inválidos."""
    p_inv = {"capacidad_x": 0, "presion": -1}
    accion_s, _ = decidir_reactivo_simple(p_inv)
    assert accion_s == "ABSTENERSE"
    estado = crear_estado_inicial()
    estado = actualizar_estado(estado, p_inv)
    accion_m, _ = decidir_reactivo_modelo(estado)
    assert accion_m == "ABSTENERSE"
    print("✅ test_percepcion_invalida OK")

# Ejecutar todos los tests
test_presion_baja()
test_primera_hora_presion_alta()
test_segunda_hora_consecutiva_presion_alta()
test_demostracion_dependencia_historia()
test_percepcion_invalida()
print("\n🎉 ¡Todos los tests pasaron!")

## Parte 3: Generar escenario real y bitácora

In [ ]:
import sys, os
# Agregar el directorio del laboratorio al path para importar el simulador
sys.path.insert(0, LAB_DIR)
os.makedirs("escenario_agente", exist_ok=True)

# Generar el escenario: zona 161 (Midtown Center), hora 8, 20 taxis, semilla 42
!python {LAB_DIR}/simulador_entorno_agente.py \
    --zona 161 \
    --hora 8 \
    --taxis-x 20 \
    --horas-historia 3 \
    --semilla 42 \
    --salida-dir escenario_agente

print("\n📁 Archivos generados:")
!ls escenario_agente/

In [ ]:
# Cargar las percepciones (lo que el agente PUEDE ver)
percepciones = pd.read_csv("escenario_agente/percepciones.csv")
print("📋 Percepciones (entrada del agente):")
percepciones[['zona', 'hora', 'taxis_x', 'demanda_total', 'demanda_x', 'capacidad_x', 'presion']]

In [ ]:
# Ejecutar ambos agentes y generar la bitácora
bitacora = procesar_secuencia(percepciones)
bitacora.to_csv("bitacora_agentes.csv", index=False)

print("📊 Bitácora comparativa:")
bitacora

## Parte 5: Informe

### 1. ¿En qué situaciones ambos agentes producen la misma acción?

- **Presión baja** (`presion < 0.85`): ambos devuelven `NO_REFORZAR`.
- **Dos o más horas consecutivas con presión alta**: el agente modelo acumula `racha >= 2` y coincide con el simple.
- En el escenario ejecutado, coinciden en las horas 7 y 8.

### 2. ¿Cuándo reaccionan de forma diferente?

En la **primera hora con presión alta**: el agente simple ya recomienda refuerzo, mientras que el basado en modelo todavía devuelve `NO_REFORZAR` (espera confirmación histórica). En el escenario, esto ocurre en la hora 6.

### 3. ¿Por qué el segundo agente está basado en modelo aunque no planifique?

Porque mantiene un **estado interno** (`racha_presion_alta`, `presion_anterior`, etc.) que representa un resumen de la historia del entorno que ya no puede percibir directamente. Según Russell & Norvig, eso es exactamente un agente basado en modelo: compensa la parcialidad de la percepción actual con un modelo del mundo. No es un planificador porque no genera ni evalúa secuencias de acciones futuras.

### 4. ¿Qué representa `tasa_otras_simulada` y qué no permite afirmar?

`tasa_otras_simulada` es una proporción **sintética y aleatoria** para distribuir viajes entre empresa X y otras ficticias. No representa datos reales de ninguna empresa, no fue estimada con datos observados, y no permite inferir la cantidad de vehículos ni la estrategia de otras empresas.

### 5. ¿Por qué `resultado_h_mas_1.csv` no puede formar parte de la percepción?

Porque contiene datos de la hora `h+1`, es decir, el **futuro** respecto al momento de decisión. Usarlo constituiría una **fuga temporal** (*data leakage*): el agente tomaría decisiones con información que no existe aún en el mundo real, haciendo la evaluación inválida.

---

### PEAS mínimo

| Elemento | Contenido |
|---|---|
| **Performance** | Recomendaciones coherentes con las reglas; ausencia de fuga temporal; abstención ante datos inválidos; trazabilidad via campo `motivo`. |
| **Environment** | Secuencia simulada zona-hora basada en datos Yellow Taxi TLC 2024; flota sintética de X; otras empresas ficticias; responsable humano que ejecuta el traslado. |
| **Actuators** | Mensajes simbólicos: `NO_REFORZAR`, `RECOMENDAR_REFUERZO`, `ABSTENERSE`. |
| **Sensors** | Lectura lógica de `percepciones.csv`; no es un sensor en tiempo real. |

---

### Limitaciones reconocidas

- Los pickups TLC son actividad Yellow Taxi **realizada**, no demanda total ni solicitudes no atendidas.
- X y "otras empresas" son **entidades ficticias**; no corresponden a empresas reales.
- La relación inversa entre flota de X y `tasa_otras_simulada` es una **hipótesis didáctica** no calibrada.
- Una unidad de capacidad por taxi-hora es una **simplificación**.
- `RECOMENDAR_REFUERZO` es un mensaje para **revisión humana**, no una orden ejecutada automáticamente.